# 🦙 Fine-tune the Support Router on Google Colab

LoRA-fine-tune a small open-source model to **be the strict-JSON router**
(`static / chitchat / rag / fallback` + entity extraction) for the Agentic
Support Orchestrator.

It trains on `router_train.jsonl` produced by `scripts/prepare_dataset.py`.

**Önce yap:** menüden **Runtime → Change runtime type → T4 GPU** seç, sonra
hücreleri yukarıdan aşağıya sırayla çalıştır.

## 0) GPU bağlı mı?

In [ ]:
!nvidia-smi

## 1) Kurulum (Unsloth + bağımlılıklar)
Birkaç dakika sürer.

In [ ]:
%%capture
!pip install unsloth
# Hata alirsan (torch._inductor / AttributeError): asagiyi ac, calistir, sonra Runtime > Restart:
# !pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo


## 2) Eğitim verisini yükle
`router_train.jsonl` **ve** `router_eval.jsonl`'i birlikte sec.

In [ ]:
from google.colab import files
import os

print("router_train.jsonl VE router_eval.jsonl'i birlikte sec:")
uploaded = files.upload()
for name in list(uploaded):
    low = name.lower()
    if "train" in low and name != "router_train.jsonl":
        os.rename(name, "router_train.jsonl")
    elif "eval" in low and name != "router_eval.jsonl":
        os.rename(name, "router_eval.jsonl")
print("train:", os.path.exists("router_train.jsonl"), "| eval:", os.path.exists("router_eval.jsonl"))


## 3) Temel modeli yükle (4-bit) + LoRA ekle
`Qwen2.5-1.5B` JSON üretiminde güçlü ve **Apache-2.0** lisanslı — router için ideal.
Maksimum hız için `unsloth/Qwen2.5-0.5B-Instruct`, çok dilli (TR) için `google/gemma-3-1b-it` deneyebilirsin.

In [ ]:
import torch
import torch._inductor.config  # Colab/torch uyumu: alt modulu onceden yukle
from unsloth import FastLanguageModel

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 🎯 Değerlendirme fonksiyonu + EĞİTİM ÖNCESİ baseline
LoRA baslangicta etkisiz (≈ ham model), bu yuzden 'fine-tune oncesi' skoru verir.

In [ ]:
import json, re, random

def _load_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]

def _pred_route(text):
    try:
        return json.loads(text).get("route"), True
    except Exception:
        m = re.search(r'"route"\s*:\s*"(\w+)"', text)
        return (m.group(1) if m else None), False

def evaluate(model, tokenizer, eval_path="router_eval.jsonl", per_route_cap=40, seed=3407):
    FastLanguageModel.for_inference(model)
    data = _load_jsonl(eval_path)
    sys_prompt = data[0]["messages"][0]["content"]
    by = {}
    for r in data:
        by.setdefault(r["route"], []).append(r)
    rng = random.Random(seed)
    sample = []
    for route, items in by.items():
        rng.shuffle(items)
        sample += items[:per_route_cap]

    correct, total, valid_json = {}, {}, 0
    for rec in sample:
        gold = rec["route"]
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": rec["messages"][1]["content"]},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")
        out = model.generate(input_ids=inputs, max_new_tokens=64, do_sample=False)
        text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
        pred, ok = _pred_route(text)
        valid_json += int(ok)
        total[gold] = total.get(gold, 0) + 1
        correct[gold] = correct.get(gold, 0) + int(pred == gold)

    accs = []
    for route in sorted(total):
        a = correct[route] / total[route]
        accs.append(a)
        print(f"  {route:9} acc={a:5.0%}  ({correct[route]}/{total[route]})")
    macro = sum(accs) / len(accs)
    print(f"  MACRO route-accuracy = {macro:.1%} | valid-JSON = {valid_json}/{len(sample)}")
    try:
        FastLanguageModel.for_training(model)
    except Exception:
        pass
    return macro

print("=== EGITIM ONCESI (baseline ~ ham model) ===")
base_macro = evaluate(model, tokenizer)


## 4) Veriyi sohbet şablonuna çevir
Her satırdaki `messages` (system/user/assistant) modelin kendi chat template'ine render edilir.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="router_train.jsonl", split="train")

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset)
print(dataset[0]["text"][:400])


## 5) Eğit (LoRA)
T4'te ~birkaç dakika. `num_train_epochs` artırırsan daha iyi öğrenir (ama overfit'e dikkat).

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",   # checkpoint kaydetme -> TRL/Unsloth pickle hatasini onler
        report_to = "none",
    ),
)

trainer.train()

## 6) Hızlı test — model artık doğru JSON + doğru route üretiyor mu?
Eğitimdeki ile **aynı** system prompt'u kullanıyoruz.

In [ ]:
import json
FastLanguageModel.for_inference(model)

sys_prompt = json.loads(open("router_train.jsonl").readline())["messages"][0]["content"]

def route(message):
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": message},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=128, do_sample=False)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

tests = [
    "How do I reset my password?",      # beklenen: rag
    "Hi there, good morning!",           # beklenen: chitchat
    "What is your refund policy?",       # beklenen: static
    "What's the weather today?",         # beklenen: fallback
]
for m in tests:
    print(m, "->", route(m))

## 🎯 Değerlendirme: fine-tuned vs baseline
`router_eval.jsonl` uzerinde once/sonra macro route-accuracy — portfolyo icin somut kanit.

In [ ]:
print("=== EGITIM SONRASI (fine-tuned) ===")
ft_macro = evaluate(model, tokenizer)
print()
print(f"ONCE  (baseline)   macro = {base_macro:.1%}")
print(f"SONRA (fine-tuned) macro = {ft_macro:.1%}")
print(f"IYILESME = {(ft_macro - base_macro) * 100:+.1f} puan")


## 7) GGUF olarak dışa aktar + indir
Bu adım llama.cpp derler (birkaç dakika). Sonunda `.gguf` dosyası bilgisayarına iner.

In [ ]:
model.save_pretrained_gguf("router-gguf", tokenizer, quantization_method="q4_k_m")

import glob
from google.colab import files
gguf = glob.glob("router-gguf/*.gguf")[0]
print("GGUF ->", gguf)
files.download(gguf)

## 8) Kendi bilgisayarında Ollama'ya yükle

İnen `.gguf` dosyasının yanına bir `Modelfile` oluştur:

```
FROM ./unsloth.Q4_K_M.gguf
PARAMETER temperature 0
```

Sonra:

```bash
ollama create support-router -f Modelfile
ollama run support-router "How do I reset my password?"
```

Projeye bağla — `.env`:

```
APP_LLM_PROVIDER=local
APP_LOCAL_ROUTER_MODEL=support-router
```

Bitti! 🎉 Artık router fallback'i senin fine-tune ettiğin model çalıştırıyor.